In [6]:
import requests, pandas as pd, numpy as np
LAT, LON = 6.9271, 79.8612            # Colombo
START, END = "2022-08-01", "2026-11-15"
common = dict(latitude=LAT, longitude=LON, start_date=START, end_date=END, timezone="Asia/Colombo")

aq = requests.get("https://air-quality-api.open-meteo.com/v1/air-quality",
        params={**common, "hourly": "pm2_5,pm10,nitrogen_dioxide,carbon_monoxide"}).json()
wx = requests.get("https://archive-api.open-meteo.com/v1/archive",
        params={**common, "hourly": "temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,wind_direction_10m"}).json()

h = pd.DataFrame(aq["hourly"]).merge(pd.DataFrame(wx["hourly"]), on="time")
h["time"] = pd.to_datetime(h["time"])
h.to_csv("colombo_hourly_raw.csv", index=False)

In [7]:
print(h.shape)
h.head()

(37632, 10)


,time,pm2_5,pm10,nitrogen_dioxide,carbon_monoxide,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,wind_direction_10m
0,2022-08-01 00:00:00,NaN,NaN,NaN,NaN,24.2,93,3.8,8.6,248
1,2022-08-01 01:00:00,NaN,NaN,NaN,NaN,23.9,94,3.1,12.0,231
2,2022-08-01 02:00:00,NaN,NaN,NaN,NaN,24.5,92,0.4,10.5,248
3,2022-08-01 03:00:00,NaN,NaN,NaN,NaN,24.9,90,0.2,12.6,250
4,2022-08-01 04:00:00,NaN,NaN,NaN,NaN,25.1,89,0.1,13.2,259


In [8]:
# Where does the PM2.5 data actually start?
first = h["pm2_5"].first_valid_index()
print("PM2.5 data starts at:", h.loc[first, "time"])

# Keep only rows from that point onwards
h = h.loc[first:].reset_index(drop=True)

print("Rows now:", h.shape)
print("Missing values left:\n", h.isna().sum())
h.head()

PM2.5 data starts at: 2022-08-04 05:00:00
Rows now: (37555, 10)
Missing values left:
 time                    0
pm2_5                   0
pm10                    0
nitrogen_dioxide        0
carbon_monoxide         0
temperature_2m          0
relative_humidity_2m    0
precipitation           0
wind_speed_10m          0
wind_direction_10m      0
dtype: int64


,time,pm2_5,pm10,nitrogen_dioxide,carbon_monoxide,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,wind_direction_10m
0,2022-08-04 05:00:00,13.6,23.8,1.8,114.0,25.0,87,2.6,22.1,251
1,2022-08-04 06:00:00,14.7,25.3,3.2,133.0,25.4,84,0.5,22.7,252
2,2022-08-04 07:00:00,14.8,25.3,5.2,160.0,26.1,84,0.1,20.5,252
3,2022-08-04 08:00:00,14.6,25.3,6.8,183.0,26.7,80,0.3,23.4,256
4,2022-08-04 09:00:00,15.4,26.5,6.3,178.0,26.8,82,2.2,24.3,246


In [9]:
# Wind direction is an angle, so turn it into u/v components before averaging
rad = np.deg2rad(h["wind_direction_10m"])
h["wind_u"] = -h["wind_speed_10m"] * np.sin(rad)
h["wind_v"] = -h["wind_speed_10m"] * np.cos(rad)

# Hourly -> daily
d = h.set_index("time").resample("D").agg({
    "pm2_5":"mean","pm10":"mean","nitrogen_dioxide":"mean","carbon_monoxide":"mean",
    "temperature_2m":"mean","relative_humidity_2m":"mean","precipitation":"sum",
    "wind_speed_10m":"mean","wind_u":"mean","wind_v":"mean"}).interpolate()

# Target = tomorrow's PM2.5
d["target"] = d["pm2_5"].shift(-1)

# Past PM2.5 values (lags) and 7-day average
for k in [1, 2, 3, 7]:
    d[f"pm25_lag{k}"] = d["pm2_5"].shift(k)
d["pm25_roll7"] = d["pm2_5"].rolling(7).mean()

# Calendar and monsoon features
d["month"] = d.index.month
d["ne_monsoon"]    = d["month"].isin([12, 1, 2]).astype(int)       # Northeast monsoon
d["inter_mon_1"]   = d["month"].isin([3, 4]).astype(int)           # First inter-monsoon
d["sw_monsoon"]    = d["month"].isin([5, 6, 7, 8, 9]).astype(int)  # Southwest monsoon
d["inter_mon_2"]   = d["month"].isin([10, 11]).astype(int)         # Second inter-monsoon
d["doy_sin"] = np.sin(2*np.pi*d.index.day_of_year/365.25)
d["doy_cos"] = np.cos(2*np.pi*d.index.day_of_year/365.25)
d["weekday"] = d.index.day_of_week

print(d.shape)
d.tail()

(1565, 24)


,pm2_5,pm10,nitrogen_dioxide,carbon_monoxide,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,wind_u,wind_v,...,pm25_lag7,pm25_roll7,month,ne_monsoon,inter_mon_1,sw_monsoon,inter_mon_2,doy_sin,doy_cos,weekday
time,,,,,,,,,,,,,,,,,,,,,
2026-11-11,16.295833,16.812500,29.275000,1426.291667,26.575000,88.333333,29.2,6.783333,-1.303167,-2.271026,...,16.104167,15.511905,11,0,0,0,1,-0.760720,0.649080,2
2026-11-12,10.020833,10.691667,21.958333,970.583333,26.108333,89.291667,15.5,9.891667,-2.029026,-9.032845,...,19.904167,14.100000,11,0,0,0,1,-0.749442,0.662070,3
2026-11-13,20.908333,24.412500,21.233333,611.166667,27.729167,78.666667,0.0,12.316667,-0.168371,-11.780360,...,13.216667,15.198810,11,0,0,0,1,-0.737943,0.674863,4
2026-11-14,18.908333,23.304167,21.820833,968.875000,26.870833,78.750000,0.0,10.016667,-0.540413,-9.708578,...,14.741667,15.794048,11,0,0,0,1,-0.726225,0.687457,5
2026-11-15,20.241667,24.954167,20.575000,953.375000,25.258333,88.833333,9.4,5.675000,-0.527903,-3.210942,...,14.812500,16.569643,11,0,0,0,1,-0.714292,0.699848,6


In [10]:
import plotly.express as px
px.box(d, x="month", y="pm2_5", title="Colombo daily PM2.5 by month").show()

In [11]:
from statsmodels.tsa.arima.model import ARIMA
import warnings; warnings.filterwarnings("ignore")

SPLIT = "2025-11-15"   # last 12 months = test set (covers every season)

# ARIMA fitted on training data only
arima = ARIMA(d.loc[d.index < SPLIT, "pm2_5"], order=(2, 0, 1)).fit()
d["arima_next"] = arima.apply(d["pm2_5"]).predict().shift(-1)   # tomorrow's forecast, made today

d = d.dropna()
FEATURES = [c for c in d.columns if c not in ("target", "arima_next")]
train, test = d[d.index < SPLIT], d[d.index >= SPLIT]
X_tr, y_tr = train[FEATURES], train["target"]
X_te, y_te = test[FEATURES], test["target"]

print("Train days:", len(train), "| Test days:", len(test))
print("Features:", FEATURES)

Train days: 1192 | Test days: 365
Features: ['pm2_5', 'pm10', 'nitrogen_dioxide', 'carbon_monoxide', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m', 'wind_u', 'wind_v', 'pm25_lag1', 'pm25_lag2', 'pm25_lag3', 'pm25_lag7', 'pm25_roll7', 'month', 'ne_monsoon', 'inter_mon_1', 'sw_monsoon', 'inter_mon_2', 'doy_sin', 'doy_cos', 'weekday']


In [12]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold

rf = lambda: RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=42, n_jobs=-1)
gb = lambda: GradientBoostingRegressor(n_estimators=400, learning_rate=0.05, max_depth=3,
                                       subsample=0.8, random_state=42)

models = {
    "Bagging – Random Forest": rf(),
    "Boosting – Gradient Boosting": gb(),
    "Stacking – RF + GB → Ridge": StackingRegressor(
        estimators=[("rf", rf()), ("gb", gb())], final_estimator=RidgeCV(), cv=KFold(5)),
}

preds = {"Baseline – Persistence (tomorrow = today)": X_te["pm2_5"].values}
train_preds = {"Baseline – Persistence (tomorrow = today)": X_tr["pm2_5"].values}

for name, m in models.items():
    print("Training:", name)
    m.fit(X_tr, y_tr)
    preds[name] = m.predict(X_te)
    train_preds[name] = m.predict(X_tr)
print("Done ✅")

Training: Bagging – Random Forest
Training: Boosting – Gradient Boosting
Training: Stacking – RF + GB → Ridge
Done ✅


In [13]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Hybrid: ARIMA forecast + RF that learns ARIMA's mistakes from the weather features
rf_res = rf().fit(X_tr, y_tr - train["arima_next"])
models["rf_residual"] = rf_res

preds["ARIMA only"] = test["arima_next"].values
train_preds["ARIMA only"] = train["arima_next"].values
preds["Hybrid – ARIMA + RF residuals"] = test["arima_next"].values + rf_res.predict(X_te)
train_preds["Hybrid – ARIMA + RF residuals"] = train["arima_next"].values + rf_res.predict(X_tr)

rows = []
for k in preds:
    rows.append(dict(Model=k,
        MAE=mean_absolute_error(y_te, preds[k]),
        RMSE=mean_squared_error(y_te, preds[k]) ** 0.5,
        R2=r2_score(y_te, preds[k]),
        Train_RMSE=mean_squared_error(y_tr, train_preds[k]) ** 0.5))
results = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)
results.round(3)

,Model,MAE,RMSE,R2,Train_RMSE
0,Hybrid – ARIMA + RF residuals,3.655,4.839,0.743,2.446
1,Bagging – Random Forest,3.724,4.890,0.737,2.498
2,Stacking – RF + GB → Ridge,3.745,4.938,0.732,2.381
3,Boosting – Gradient Boosting,3.761,5.036,0.721,2.407
4,ARIMA only,3.798,5.056,0.719,5.055
5,Baseline – Persistence (tomorrow = today),3.848,5.245,0.698,5.300


In [14]:
imp = pd.Series(models["Bagging – Random Forest"].feature_importances_, FEATURES).sort_values()
px.bar(imp.tail(12), orientation="h", title="Random Forest – top 12 features").show()

In [15]:
import joblib
joblib.dump({"models": models, "features": FEATURES}, "models.pkl")
results.to_csv("results.csv", index=False)
pd.DataFrame(preds, index=test.index).assign(actual=y_te).to_csv("test_predictions.csv")
d.to_csv("colombo_daily.csv")
print("Saved ✅")

Saved ✅
